# Vehicle Type Classification — Logistic Regression

Runs end to end and writes `submissions/submission_logistic_regression.csv`.

Linear decision boundaries (softmax over a linear score). Regularisation strength C is tuned. Best of the four models in CV.

## Setup and preprocessing
* Drop `vehicle_id` (identifier only).
* Drop the 2,500 exact duplicate training rows: they would leak across CV folds and inflate scores (labels never conflict, so nothing is lost).
* No imputation (no missing values), no outlier removal (bounded values), no resampling (balanced classes).
* Standardise features inside the pipeline (needed for regularised logistic regression; fitted per fold, so no leakage).

In [1]:
import os, json, numpy as np, pandas as pd, warnings
from sklearn.model_selection import StratifiedKFold, RepeatedStratifiedKFold, GridSearchCV, RandomizedSearchCV, cross_val_score, cross_val_predict
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, f1_score
warnings.filterwarnings("ignore")
os.makedirs("submissions", exist_ok=True)
SEED = 42; np.random.seed(SEED)
for d in ["data", ".", "..", "/kaggle/input/vehicle-type-classification"]:
    if os.path.exists(f"{d}/vehicle_train_10000.csv"): DATA_DIR = d; break
else: raise FileNotFoundError("Put the competition CSVs next to this notebook or in ./data")
train  = pd.read_csv(f"{DATA_DIR}/vehicle_train_10000.csv")
test   = pd.read_csv(f"{DATA_DIR}/vehicle_test.csv")
sample = pd.read_csv(f"{DATA_DIR}/vehicle_sample_submission.csv")
TARGET, ID = "vehicle_class", "vehicle_id"
FEATURES = [c for c in test.columns if c != ID]
df = train.drop_duplicates(subset=FEATURES).reset_index(drop=True)   # remove 2,500 exact duplicate rows
X, y, X_test = df[FEATURES], df[TARGET].values, test[FEATURES]
CV5 = StratifiedKFold(5, shuffle=True, random_state=SEED)
print("training rows after de-dup:", len(df), "| class counts:", np.bincount(y))

training rows after de-dup: 7500 | class counts: [2539 2490 2471]


## Hyper-parameter tuning (5-fold stratified CV, accuracy)

In [2]:
pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000, random_state=SEED))
search = GridSearchCV(pipe, {"logisticregression__C": [0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1, 3, 10, 100]}, cv=CV5, scoring="accuracy").fit(X, y)
print(pd.DataFrame(search.cv_results_)[["param_logisticregression__C", "mean_test_score", "std_test_score"]].round(4).to_string(index=False))

 param_logisticregression__C  mean_test_score  std_test_score
                       0.001           0.8695          0.0069
                       0.003           0.8821          0.0070
                       0.010           0.8909          0.0055
                       0.030           0.8937          0.0068
                       0.100           0.8941          0.0054
                       0.300           0.8943          0.0055
                       1.000           0.8941          0.0057
                       3.000           0.8940          0.0052
                      10.000           0.8940          0.0052
                     100.000           0.8939          0.0054


## Stability check, overfitting check, and submission

In [3]:
best = search.best_estimator_
print("best params:", search.best_params_, "| 5-fold CV acc:", round(search.best_score_, 4))
rep = cross_val_score(best, X, y, cv=RepeatedStratifiedKFold(n_splits=5, n_repeats=6, random_state=SEED + 1), scoring="accuracy", n_jobs=1)
oof = cross_val_predict(best, X, y, cv=CV5)
print(f"Repeated CV accuracy: {rep.mean():.4f} ± {rep.std():.4f} | OOF macro-F1: {f1_score(y, oof, average='macro'):.4f}")
best.fit(X, y)
print("train accuracy:", round(accuracy_score(y, best.predict(X)), 4), "vs CV", round(search.best_score_, 4))
pred = best.predict(X_test).astype(int)
sub = pd.DataFrame({ID: test[ID], TARGET: pred})
assert list(sub.columns) == list(sample.columns) and len(sub) == len(sample) and (sub[ID].values == sample[ID].values).all() and set(sub[TARGET]) <= {0, 1, 2}
sub.to_csv("submissions/submission_logistic_regression.csv", index=False)
json.dump(dict(model="Logistic Regression", cv_acc=float(search.best_score_), repeated_cv=float(rep.mean()), repeated_cv_std=float(rep.std()),
               macro_f1=float(f1_score(y, oof, average="macro")), train_acc=float(accuracy_score(y, best.predict(X))), params=str(search.best_params_)), open("submissions/submission_logistic_regression_result.json", "w"))
print("written submission_logistic_regression.csv | predicted class counts:", np.bincount(pred, minlength=3).tolist())

best params: {'logisticregression__C': 0.3} | 5-fold CV acc: 0.8943


Repeated CV accuracy: 0.8934 ± 0.0083 | OOF macro-F1: 0.8946
train accuracy: 0.8953 vs CV 0.8943
written submission_logistic_regression.csv | predicted class counts: [839, 832, 829]
